In [1]:
import qrcode
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import filedialog, messagebox

# Function to generate QR Code
def generate_qr():
    data = entry_data.get().strip()
    
    if data == "":
        messagebox.showerror("Error", "Please enter text or URL!")
        return
    
    try:
        # Create QR Code with high error correction
        qr = qrcode.QRCode(
            version=None,
            error_correction=qrcode.constants.ERROR_CORRECT_H,
            box_size=10,
            border=4,
        )
        
        qr.add_data(data)
        qr.make(fit=True)
        
        qr_img = qr.make_image(fill_color="black", back_color="white").convert("RGB")
        
        # If logo is selected
        if logo_path.get() != "":
            logo = Image.open(logo_path.get())
            
            # Resize logo
            qr_width, qr_height = qr_img.size
            logo_size = qr_width // 4
            logo = logo.resize((logo_size, logo_size))
            
            # Position logo at center
            pos = ((qr_width - logo_size) // 2, (qr_height - logo_size) // 2)
            
            # Paste logo
            qr_img.paste(logo, pos, mask=logo if logo.mode == 'RGBA' else None)
        
        # Save generated image in memory
        global final_qr
        final_qr = qr_img
        
        # Display in GUI
        img_tk = ImageTk.PhotoImage(qr_img.resize((250, 250)))
        label_qr.config(image=img_tk)
        label_qr.image = img_tk
        
        messagebox.showinfo("Success", "QR Code Generated Successfully!")
    
    except Exception as e:
        messagebox.showerror("Error", str(e))


# Function to save QR Code
def save_qr():
    if final_qr is None:
        messagebox.showwarning("Warning", "Generate QR Code first!")
        return
    
    file_path = filedialog.asksaveasfilename(
        defaultextension=".png",
        filetypes=[("PNG files", "*.png")]
    )
    
    if file_path:
        final_qr.save(file_path)
        messagebox.showinfo("Saved", f"QR Code saved as:\n{file_path}")


# Function to select logo
def select_logo():
    path = filedialog.askopenfilename(
        filetypes=[("Image Files", "*.png *.jpg *.jpeg")]
    )
    if path:
        logo_path.set(path)
        label_logo.config(text="Logo Selected ✔")


# Create main window
root = tk.Tk()
root.title("QR Code Generator with Logo")
root.geometry("400x550")
root.resizable(False, False)

final_qr = None
logo_path = tk.StringVar()

# Title
title = tk.Label(root, text="QR Code Generator", font=("Arial", 18, "bold"))
title.pack(pady=10)

# Input Label
label_input = tk.Label(root, text="Enter Text or URL:")
label_input.pack()

# Input Entry
entry_data = tk.Entry(root, width=40)
entry_data.pack(pady=5)

# Select Logo Button
btn_logo = tk.Button(root, text="Select Logo (Optional)", command=select_logo)
btn_logo.pack(pady=10)

label_logo = tk.Label(root, text="No Logo Selected")
label_logo.pack()

# Generate Button
btn_generate = tk.Button(root, text="Generate QR Code", command=generate_qr, bg="green", fg="white", width=20)
btn_generate.pack(pady=15)

# QR Display Label
label_qr = tk.Label(root)
label_qr.pack(pady=10)

# Save Button
btn_save = tk.Button(root, text="Save QR Code", command=save_qr, bg="blue", fg="white", width=20)
btn_save.pack(pady=10)

# Run the GUI
root.mainloop()
